# Week 8 Homework — Custom ReAct Agent: New Domain, New Tools
**COM SCI-X 455 · Domain: AI and video-game research papers (arXiv)** · Push to `genai-course/week8/homework/`

In [2]:
!pip install langgraph langchain-openai langchain-community duckduckgo-search requests --quiet
!pip install -U ddgs --quiet

In [3]:
import os, json, math, time, re, operator
import xml.etree.ElementTree as ET
from datetime import date
from typing import TypedDict, Annotated
import requests
from statistics import mean

from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from langgraph.graph import StateGraph, END
from langgraph.errors import GraphRecursionError
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_community.tools import DuckDuckGoSearchRun

print("All imports OK")

All imports OK


/tmp/ipykernel_549/2223558762.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## Part 1 — Domain-Specific Agent with 3 Tools
Domain: **AI and video-game research papers on arXiv.**
- `arxiv_search` — custom, real call to the arXiv API
- `metric_calc` — custom, domain computations (percent change, frame time, mean, days between dates, ratio)
- `web_search` — DuckDuckGo (built-in search) wrapped with a domain docstring

In [4]:
ARXIV_URL = "https://export.arxiv.org/api/query"

_ATOM = {
    "a": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom"
}

_last_arxiv_call = [0.0]


@tool
def arxiv_search(query: str, max_results: int = 5,
                 sort_by: str = "relevance", category: str = "") -> str:
    """Search arXiv for research papers related to AI and video games.

    Returns paper details such as the title, authors, date, category,
    and abstract. Results are returned as a JSON list.
    """
    terms = [t.strip('"\'') for t in query.split() if t.strip('"\'')]

    if not terms:
        return "Please enter a search query."

    search_query = " AND ".join(f"all:{t}" for t in terms)

    if category:
        search_query += f" AND cat:{category}"

    params = {
        "search_query": search_query,
        "start": 0,
        "max_results": max(1, min(int(max_results), 10)),
        "sortBy": "submittedDate" if sort_by == "date" else "relevance",
        "sortOrder": "descending"
    }

    # Wait before making another request to avoid sending requests too quickly.
    wait = 3.0 - (time.time() - _last_arxiv_call[0])

    if wait > 0:
        time.sleep(wait)

    try:
        response = requests.get(ARXIV_URL, params=params, timeout=30)
        _last_arxiv_call[0] = time.time()
        response.raise_for_status()
        root = ET.fromstring(response.text)

    except Exception as e:
        return f"Error searching arXiv: {e}"

    papers = []

    for entry in root.findall("a:entry", _ATOM):
        authors = [
            author.findtext("a:name", default="", namespaces=_ATOM)
            for author in entry.findall("a:author", _ATOM)
        ]

        primary_category = entry.find("arxiv:primary_category", _ATOM)

        paper = {
            "arxiv_id": entry.findtext(
                "a:id", default="", namespaces=_ATOM
            ).split("/abs/")[-1],

            "title": " ".join(
                entry.findtext("a:title", default="", namespaces=_ATOM).split()
            ),

            "published": entry.findtext(
                "a:published", default="", namespaces=_ATOM
            )[:10],

            "n_authors": len(authors),
            "authors": authors[:3],

            "primary_category": (
                primary_category.get("term")
                if primary_category is not None else ""
            ),

            "abstract": " ".join(
                entry.findtext("a:summary", default="", namespaces=_ATOM).split()
            )[:600]
        }

        papers.append(paper)

    if not papers:
        return "No papers found for this search."

    return json.dumps(papers)


def _metric_impl(operation, values):
    """Calculate one of the supported metrics."""
    operation = operation.strip().lower()
    parts = [p.strip() for p in values.split(",") if p.strip()]

    try:
        if operation == "days_between":
            if len(parts) != 2:
                return "Please provide two dates in YYYY-MM-DD format."

            date1, date2 = [date.fromisoformat(p) for p in parts]
            return str(abs((date2 - date1).days))

        numbers = [float(p) for p in parts]

        if operation == "pct_change":
            if len(numbers) != 2 or numbers[0] == 0:
                return "Please provide an old and new value. The old value cannot be zero."

            result = (numbers[1] - numbers[0]) / numbers[0] * 100
            return str(round(result, 4))

        elif operation == "frame_time_ms":
            if len(numbers) != 1 or numbers[0] <= 0:
                return "Please enter one positive FPS value."

            return str(round(1000 / numbers[0], 4))

        elif operation == "fps_from_ms":
            if len(numbers) != 1 or numbers[0] <= 0:
                return "Please enter one positive frame time in milliseconds."

            return str(round(1000 / numbers[0], 4))

        elif operation == "mean":
            if not numbers:
                return "Please enter at least one number."

            return str(round(sum(numbers) / len(numbers), 4))

        elif operation == "ratio":
            if len(numbers) != 2 or numbers[1] == 0:
                return "Please provide two numbers, and the second cannot be zero."

            return str(round(numbers[0] / numbers[1], 4))

        else:
            return "Unknown operation. Try pct_change, frame_time_ms, fps_from_ms, mean, days_between, or ratio."

    except Exception as e:
        return f"Could not calculate the metric: {e}"


@tool
def metric_calc(operation: str, values: str) -> str:
    """Calculate basic metrics used in research and game performance analysis.

    Supported operations include percentage change, frame time,
    FPS, mean, date differences, and ratios.
    """
    return _metric_impl(operation, values)


# Set up the web search tool.
_ddg = DuckDuckGoSearchRun()


@tool
def web_search(query: str) -> str:
    """Search the web for information about games, companies,
    gaming benchmarks, and other topics outside arXiv.
    """
    try:
        return _ddg.run(query)

    except Exception as e:
        return f"Web search error: {e}"


# Tools available to the agent.
TOOLS = [arxiv_search, metric_calc, web_search]

TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}

print([tool.name for tool in TOOLS])


['arxiv_search', 'metric_calc', 'web_search']


In [5]:
# ── State with working memory ─────────────────────────────────────────────
class DomainState(TypedDict):
    messages:   Annotated[list, operator.add]
    step_count: int                                   # number of tool calls requested so far
    findings:   Annotated[list[dict], operator.add]   # domain working memory: papers, computations, web notes

MAX_STEPS = 12

SYSTEM_PROMPT = (
    "You are a research assistant for AI and video-game research papers on arXiv. "
    "Use arxiv_search to find papers, metric_calc for EVERY calculation (never compute in your head), "
    "and web_search only for context that is not on arXiv. "
    "Ground every claim in tool observations and cite arXiv IDs. "
    "If you cannot verify something with your tools, say so."
)

def summarize_findings(findings, max_items=15):
    papers, comps, webs, seen = [], [], [], set()
    for f in findings:
        if f["kind"] == "paper" and f["arxiv_id"] not in seen:
            seen.add(f["arxiv_id"])
            papers.append(f"- paper {f['arxiv_id']} | {f['title'][:80]} | {f['published']} | {f['n_authors']} authors")
        elif f["kind"] == "computation":
            comps.append(f"- {f['op']}({f['inputs']}) = {f['result']}")
        elif f["kind"] == "web":
            webs.append(f"- web '{f['query']}': {f['snippet'][:100]}")
    lines = (papers + comps + webs)[:max_items]
    return "\n".join(lines) if lines else "(nothing yet)"

def extract_findings(call, observation):
    name, args = call["name"], call["args"]
    if name == "arxiv_search":
        try:
            return [{"kind": "paper", "arxiv_id": p["arxiv_id"], "title": p["title"],
                     "published": p["published"], "n_authors": p["n_authors"]}
                    for p in json.loads(observation)]
        except Exception:
            return []
    if name == "metric_calc":
        if str(observation).startswith("Metric error"):
            return []
        return [{"kind": "computation", "op": args.get("operation"), "inputs": args.get("values"),
                 "result": str(observation)}]
    if name == "web_search":
        return [{"kind": "web", "query": args.get("query"), "snippet": str(observation)[:200]}]
    return []

llm = ChatOpenAI(model="gpt-4o", temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)

def reason_node(state):
    memory = summarize_findings(state["findings"])
    system = SystemMessage(content=f"{SYSTEM_PROMPT}\n\nWORKING MEMORY (findings so far):\n{memory}")
    response = llm_with_tools.invoke([system] + list(state["messages"]))
    n_calls = len(response.tool_calls) if getattr(response, "tool_calls", None) else 0
    return {"messages": [response], "step_count": state["step_count"] + n_calls}

def act_node(state):
    last = state["messages"][-1]
    out_msgs, new_findings = [], []
    for call in last.tool_calls:
        try:
            obs = TOOLS_BY_NAME[call["name"]].invoke(call["args"])
        except Exception as e:
            obs = f"Tool error: {e}"
        out_msgs.append(ToolMessage(content=str(obs), tool_call_id=call["id"], name=call["name"]))
        new_findings.extend(extract_findings(call, str(obs)))
    return {"messages": out_msgs, "findings": new_findings}

def should_continue(state):
    if state["step_count"] >= MAX_STEPS:
        return "end"
    last = state["messages"][-1]
    return "call_tool" if getattr(last, "tool_calls", None) else "end"

graph = StateGraph(DomainState)
graph.add_node("reason", reason_node)
graph.add_node("tool", act_node)
graph.set_entry_point("reason")
graph.add_conditional_edges("reason", should_continue, {"call_tool": "tool", "end": END})
graph.add_edge("tool", "reason")
app = graph.compile()

print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reason(reason)
	tool(tool)
	__end__([<p>__end__</p>]):::last
	__start__ --> reason;
	reason -. &nbsp;end&nbsp; .-> __end__;
	reason -. &nbsp;call_tool&nbsp; .-> tool;
	tool --> reason;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [6]:
def initial_state(task):
    return {"messages": [HumanMessage(content=task)], "step_count": 0, "findings": []}

def tool_calls_in(messages):
    return [tc for m in messages if getattr(m, "tool_calls", None) for tc in m.tool_calls]

def run_task(task, recursion_limit=50):
    final = app.invoke(initial_state(task), config={"recursion_limit": recursion_limit})
    return {"task": task,
            "steps": len(tool_calls_in(final["messages"])),
            "step_count": final["step_count"],
            "answer": final["messages"][-1].content,
            "findings": final["findings"],
            "messages": final["messages"]}

def print_trace(r, title=""):
    print("=" * 70)
    print(title, "|", r["task"])
    print("=" * 70)
    for m in r["messages"]:
        if m.type == "human":
            continue
        if m.type == "ai":
            if m.content:
                print(f"THOUGHT/ANSWER: {m.content}")
            for tc in (getattr(m, "tool_calls", None) or []):
                print(f"ACTION: {tc['name']}({tc['args']})")
        elif m.type == "tool":
            print(f"OBSERVATION: {str(m.content)[:300]}")
    print(f"--> tool calls: {r['steps']} | step_count: {r['step_count']} | findings stored: {len(r['findings'])}")
    print()

# 3-step research task in the domain (search -> search/compute -> compute -> answer)
demo = run_task("Find the two most recent arXiv papers on LLM-driven NPCs in games and tell me how many days apart they were published.")
print_trace(demo, "DEMO")
print("Findings in state:")
print(summarize_findings(demo["findings"]))

DEMO | Find the two most recent arXiv papers on LLM-driven NPCs in games and tell me how many days apart they were published.
ACTION: arxiv_search({'query': 'LLM-driven NPCs in games', 'max_results': 2, 'sort_by': 'submittedDate', 'category': 'cs.AI'})
OBSERVATION: No papers found for this search.
THOUGHT/ANSWER: I couldn't find any recent arXiv papers specifically on "LLM-driven NPCs in games." It's possible that this topic hasn't been explicitly covered in the titles or abstracts of recent submissions. You might want to try broader search terms or check other sources for the latest research in this area.
--> tool calls: 1 | step_count: 1 | findings stored: 0

Findings in state:
(nothing yet)


## Part 2 — Evaluation on 10 Domain Tasks
4 simple (1–2 tool calls) · 4 multi-step (3–5 tool calls) · 2 adversarial.
`Optimal` = the smallest sensible number of tool calls for the task.

In [7]:
TASKS = [
    # ── simple lookups
    {"id": "H1", "type": "simple", "optimal": 1,
     "task": "Find one arXiv paper about reinforcement learning for StarCraft II and give its title, first three authors and arXiv ID."},
    {"id": "H2", "type": "simple", "optimal": 1,
     "task": "What is the arXiv ID and publication date of the most recent paper about procedural content generation with large language models?"},
    {"id": "H3", "type": "simple", "optimal": 1,
     "task": "A game targets 144 FPS. What is its frame-time budget in milliseconds?"},
    {"id": "H4", "type": "simple", "optimal": 1,
     "task": "Find one arXiv paper on Monte Carlo Tree Search for games and summarise its main claim in one sentence."},
    # ── multi-step
    {"id": "H5", "type": "multi-step", "optimal": 3,
     "task": "Find the most recent arXiv paper on reinforcement learning in Minecraft and the most recent one on reinforcement learning in Dota 2. Report which is newer and by how many days."},
    {"id": "H6", "type": "multi-step", "optimal": 4,
     "task": "Find the newest arXiv paper on each of these topics: 'game playing agents', 'procedural content generation', 'NPC dialogue'. Then compute the mean number of authors across the three papers."},
    {"id": "H7", "type": "multi-step", "optimal": 4,
     "task": "Find an arXiv paper on diffusion models that simulate a game engine. Compute the frame time in ms at 20 FPS and at 60 FPS, then the percent change in frame time going from 20 FPS to 60 FPS."},
    {"id": "H8", "type": "multi-step", "optimal": 3,
     "task": "Find an arXiv paper on StarCraft agents. Then compute the relative percent improvement of a win rate going from 42% to 61%, and the average of the two win rates, and report both with the paper's arXiv ID."},
    # ── adversarial
    {"id": "H9", "type": "adversarial (unanswerable)", "optimal": 2,
     "task": "What was the exact training compute, in GPU-hours, of the proprietary NPC dialogue system used in Cyberpunk 2077?"},
    {"id": "H10", "type": "adversarial (failure trigger: unbounded search)", "optimal": 1,
     "task": "Count exactly how many arXiv papers have 'Minecraft' in the title. Do not stop until the count is exact."},
]


In [8]:
runs = {}
for t in TASKS:
    r = None
    for attempt in range(2):                      # retry once on network errors
        try:
            r = run_task(t["task"]); r["error"] = None
            break
        except GraphRecursionError as e:
            r = {"task": t["task"], "steps": MAX_STEPS, "step_count": MAX_STEPS,
                 "answer": f"[RECURSION LIMIT: {e}]", "findings": [], "error": None,
                 "messages": [HumanMessage(content=t["task"])]}
            break
        except Exception as e:
            r = {"task": t["task"], "steps": 0, "step_count": 0, "answer": f"[ERROR: {e}]",
                 "findings": [], "error": str(e), "messages": [HumanMessage(content=t["task"])]}
            time.sleep(5)
    runs[t["id"]] = r
    print_trace(r, t["id"])

errored = [k for k, r in runs.items() if r["error"]]
print("Tasks that errored:", errored)

H1 | Find one arXiv paper about reinforcement learning for StarCraft II and give its title, first three authors and arXiv ID.
ACTION: arxiv_search({'query': 'reinforcement learning StarCraft II', 'max_results': 1})
OBSERVATION: [{"arxiv_id": "1708.04782v1", "title": "StarCraft II: A New Challenge for Reinforcement Learning", "published": "2017-08-16", "n_authors": 25, "authors": ["Oriol Vinyals", "Timo Ewalds", "Sergey Bartunov"], "primary_category": "cs.LG", "abstract": "This paper introduces SC2LE (StarCraft II Learning 
THOUGHT/ANSWER: The arXiv paper about reinforcement learning for StarCraft II is titled "StarCraft II: A New Challenge for Reinforcement Learning." The first three authors are Oriol Vinyals, Timo Ewalds, and Sergey Bartunov. The arXiv ID is 1708.04782v1.
--> tool calls: 1 | step_count: 1 | findings stored: 1

H2 | What is the arXiv ID and publication date of the most recent paper about procedural content generation with large language models?
ACTION: arxiv_search({'q

In [9]:
def grounding_hint(r):
    obs = " ".join(str(m.content) for m in r["messages"] if m.type == "tool")
    ans = str(r["answer"])
    ids = set(re.findall(r"\d{4}\.\d{4,5}", ans))
    nums = set(re.findall(r"\d+(?:\.\d+)?", ans)) - set(re.findall(r"\d+(?:\.\d+)?", r["task"]))
    return {"tool_calls": r["steps"],
            "arxiv_ids_not_in_observations": sorted(i for i in ids if i not in obs),
            "numbers_not_in_observations": sorted(n for n in nums if n not in obs)}

for t in TASKS:
    print(t["id"], grounding_hint(runs[t["id"]]))

H1 {'tool_calls': 1, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H2 {'tool_calls': 1, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H3 {'tool_calls': 2, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H4 {'tool_calls': 1, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H5 {'tool_calls': 3, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H6 {'tool_calls': 4, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H7 {'tool_calls': 6, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': ['66.67']}
H8 {'tool_calls': 3, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H9 {'tool_calls': 1, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}
H10 {'tool_calls': 6, 'arxiv_ids_not_in_observations': [], 'numbers_not_in_observations': []}


In [10]:
MANUAL = {
    "H1":  {"quality": 1,   "failure_mode": "none", "hallucinated": False},
    "H2":  {"quality": 0.5, "failure_mode": "none", "hallucinated": False},
    "H3":  {"quality": 1,   "failure_mode": "none", "hallucinated": False},
    "H4":  {"quality": 1,   "failure_mode": "none", "hallucinated": False},
    "H5":  {"quality": 0.5, "failure_mode": "tool argument hallucination", "hallucinated": False},
    "H6":  {"quality": 1,   "failure_mode": "none", "hallucinated": False},
    "H7":  {"quality": 1,   "failure_mode": "none", "hallucinated": False},
    "H8":  {"quality": 0.5, "failure_mode": "tool argument hallucination", "hallucinated": False},
    "H9":  {"quality": 1,   "failure_mode": "none", "hallucinated": False},
    "H10": {"quality": 0.5, "failure_mode": "premature termination", "hallucinated": True},
}


In [11]:
assert not [k for k, r in runs.items() if r["error"]], "Some tasks errored (network). Re-run them before grading."
assert all(v is not None for g in MANUAL.values() for v in g.values()), "Fill MANUAL above, then re-run this cell."

rows = []
for t in TASKS:
    r, g = runs[t["id"]], MANUAL[t["id"]]
    steps = r["steps"]
    rows.append({"task_id": t["id"], "steps": steps, "optimal": t["optimal"],
                 "efficiency": round(t["optimal"] / max(steps, 1), 2),
                 "quality": g["quality"], "failure_mode": g["failure_mode"],
                 "hallucinated": "Yes" if g["hallucinated"] else "No"})

task_success_rate   = sum(r["quality"] for r in rows) / len(rows)
avg_step_efficiency = mean(r["efficiency"] for r in rows)
hallucination_count = sum(r["hallucinated"] == "Yes" for r in rows)

header = "| Task | Steps | Optimal | Efficiency | Quality | Failure mode | Hallucinated? |\n|---|---|---|---|---|---|---|\n"
table_md = header + "\n".join(
    f"| {r['task_id']} | {r['steps']} | {r['optimal']} | {r['efficiency']} | {r['quality']} | {r['failure_mode']} | {r['hallucinated']} |"
    for r in rows)
summary_md = (f"\n\n**task_success_rate:** {task_success_rate:.2f}  \n"
              f"**avg_step_efficiency:** {avg_step_efficiency:.2f}  \n"
              f"**hallucination_count:** {hallucination_count}\n")
open("evaluation_table.md", "w").write("# Evaluation table\n\n" + table_md + summary_md)
print(table_md + summary_md)

| Task | Steps | Optimal | Efficiency | Quality | Failure mode | Hallucinated? |
|---|---|---|---|---|---|---|
| H1 | 1 | 1 | 1.0 | 1 | none | No |
| H2 | 1 | 1 | 1.0 | 0.5 | none | No |
| H3 | 2 | 1 | 0.5 | 1 | none | No |
| H4 | 1 | 1 | 1.0 | 1 | none | No |
| H5 | 3 | 3 | 1.0 | 0.5 | tool argument hallucination | No |
| H6 | 4 | 4 | 1.0 | 1 | none | No |
| H7 | 6 | 4 | 0.67 | 1 | none | No |
| H8 | 3 | 3 | 1.0 | 0.5 | tool argument hallucination | No |
| H9 | 1 | 2 | 2.0 | 1 | none | No |
| H10 | 6 | 1 | 0.17 | 0.5 | premature termination | Yes |

**task_success_rate:** 0.80  
**avg_step_efficiency:** 0.93  
**hallucination_count:** 1



### Analysis (200 words)
*Write after reading the table: What is the dominant failure mode in your domain-specific agent? Is it different from your general-purpose Lab 7 agent? Why?*

Analysis

The main problem with my domain specific agent was using the wrong tool arguments. In H5 and H8, the agent used operation names that didn't exist in the calculator so the calculations failed.

This was different from my Lab 7 agent where all the calculator calls worked correctly. I think this is because the Lab 7 calculator accepts normal mathematical expressions such as 42000 * 0.0825, which are easier for the AI to understand. However, the calculator in my current project requires exact operation names and specific input formats, making it easier for the agent to make mistakes.

There were also other problems. In H2, the agent found a research paper that didn't really match the topic. In H10, it claimed there were exactly 10 Minecraft papers, even though the search tool could only return 10 results at a time. This shows that the agent sometimes gives confident answers without having enough information.

My Lab 7 agent had a different issue which was getting stuck in a loop when a search didn't return any results. In comparison, my current agent usually stopped but sometimes gave incorrect or unsupported answers.

To improve the agent I would make the tool instructions clearer by listing the correct operation names and improve how it handles searches with limited results.